# Gold Layer - Implementation Summary

## Three Core Tables Built

1. **Gold Daily Market Summary** - Daily electricity price aggregations
2. **Gold Weather Impact Summary** - Daily weather conditions with risk scoring
3. **Gold Grid Incident Summary** - Daily incident aggregations by severity

## Data Sources (Silver Layer)

* `silver_market_prices` (826 records) - Hourly electricity prices
* `silver_weather` (1,140 records) - Hourly weather observations
* `silver_grid_events` (165 events) - Grid operational incidents

**Coverage:** January 1-15, 2024 | **Regions:** DK, FI, NO, SE

## Build Status

✅ **Table 1:** `gold_daily_market_summary` (60 records)
✅ **Table 2:** `gold_weather_impact_summary` (60 records)
✅ **Table 3:** `gold_grid_incident_summary` (97 records)

## 1. Gold Daily Market Summary

**Purpose:** Daily electricity price aggregations by region

**Source:** `silver_market_prices` (hourly data)

**Business Question:** "What is the daily market situation by region?"

### Implementation

```sql
-- Aggregate hourly prices to daily level
SELECT 
  report_day,
  region,
  AVG(price_eur_per_mwh) as avg_price_eur_mwh,
  MAX(price_eur_per_mwh) as max_price_eur_mwh,
  MIN(price_eur_per_mwh) as min_price_eur_mwh,
  SUM(volume_mwh) as total_volume_mwh,
  COUNT(CASE WHEN price_eur_per_mwh > regional_p90 THEN 1 END) as high_price_hours,
  STDDEV(price_eur_per_mwh) as price_volatility_stddev,
  COUNT(*) as hourly_observations
FROM silver_market_prices
GROUP BY report_day, region
```

### Output: `gold_daily_market_summary`

**Table:** `vattenfall_dev.gold.gold_daily_market_summary`

**Grain:** `report_day` × `region` (60 records)

**Key Columns:**
* `report_day`, `region`
* `avg_price_eur_mwh` - Daily average price
* `max_price_eur_mwh`, `min_price_eur_mwh` - Price range
* `total_volume_mwh` - Total trading volume
* `high_price_hours` - Hours above P90 threshold
* `price_volatility_stddev` - Intraday price volatility
* `hourly_observations` - Data completeness check

### Key Findings

**Regional Averages (Jan 1-15):**
* Finland: 48.62 EUR/MWh (highest - 9% premium)
* Sweden: 44.98 EUR/MWh
* Denmark: 44.40 EUR/MWh
* Norway: 44.40 EUR/MWh (lowest)

**Peak Day:** January 9, Finland at 56.66 EUR/MWh

**Most Volatile:** January 12, Finland with 14.55 stddev

## 2. Gold Weather Impact Summary

**Purpose:** Daily weather conditions with risk scoring by region

**Source:** `silver_weather` (hourly data aggregated to daily)

**Business Question:** "What weather conditions may affect operations by day and region?"

### Implementation

```sql
-- Aggregate hourly weather to daily + calculate risk scores
SELECT 
  report_day,
  region,
  AVG(temperature_c) as avg_temperature_c,
  AVG(wind_speed_ms) as avg_wind_speed_ms,
  SUM(precipitation_mm) as total_precipitation_mm,
  AVG(cloud_cover_percent) as avg_cloud_cover_pct,
  
  -- Risk scoring
  CASE
    WHEN AVG(temperature_c) <= -4 THEN 3
    WHEN AVG(temperature_c) <= -2 THEN 2
    WHEN AVG(temperature_c) <= 0 THEN 1
    ELSE 0
  END as temp_risk_score,
  
  CASE
    WHEN AVG(wind_speed_ms) >= 12 THEN 3
    WHEN AVG(wind_speed_ms) >= 10 THEN 2
    WHEN AVG(wind_speed_ms) >= 8 THEN 1
    ELSE 0
  END as wind_risk_score,
  
  CASE
    WHEN SUM(precipitation_mm) >= 5 THEN 3
    WHEN SUM(precipitation_mm) >= 2 THEN 2
    WHEN SUM(precipitation_mm) >= 0.5 THEN 1
    ELSE 0
  END as precip_risk_score,
  
  (temp_risk + wind_risk + precip_risk) as weather_risk_score,
  
  -- Finland adjustment (most weather-sensitive)
  (temp_risk + wind_risk + precip_risk) * 
    CASE WHEN region = 'FI' THEN 1.2 ELSE 1.0 END as adjusted_risk_score,
  
  -- Risk level
  CASE
    WHEN adjusted_risk_score >= 7 THEN 'EXTREME'
    WHEN adjusted_risk_score >= 5 THEN 'HIGH'
    WHEN adjusted_risk_score >= 3 THEN 'ELEVATED'
    WHEN adjusted_risk_score >= 1 THEN 'MODERATE'
    ELSE 'LOW'
  END as weather_risk_level,
  
  COUNT(CASE WHEN any_risk > 0 THEN 1 END) as extreme_weather_hours
FROM silver_weather
GROUP BY report_day, region
```

### Output: `gold_weather_impact_summary`

**Table:** `vattenfall_dev.gold.gold_weather_impact_summary`

**Grain:** `report_day` × `region` (60 records)

**Key Columns:**
* `report_day`, `region`
* `avg_temperature_c`, `avg_wind_speed_ms`, `total_precipitation_mm`, `avg_cloud_cover_pct`
* `temp_risk_score` (0-3), `wind_risk_score` (0-3), `precip_risk_score` (0-3)
* `weather_risk_score` (0-9), `adjusted_risk_score` (with FI 1.2× multiplier)
* `weather_risk_level` (LOW/MODERATE/ELEVATED/HIGH/EXTREME)
* `extreme_weather_hours` - Count of hours with risk conditions

### Risk Thresholds (from Silver Layer Analysis)

**Critical Finding:** ALL incidents occurred with wind ≥ 8 m/s

**Temperature Risk:**
* ≤ -4°C = 3 points (extreme cold)
* ≤ -2°C = 2 points (severe cold)
* ≤ 0°C = 1 point (freezing)

**Wind Risk:**
* ≥ 12 m/s = 3 points (storm force)
* ≥ 10 m/s = 2 points (incident threshold)
* ≥ 8 m/s = 1 point (elevated risk)

**Precipitation Risk:**
* ≥ 5 mm = 3 points (heavy)
* ≥ 2 mm = 2 points (incident threshold)
* ≥ 0.5 mm = 1 point (light)

## 3. Gold Grid Incident Summary

**Purpose:** Daily incident aggregations by region and severity band

**Source:** `silver_grid_events` (event-level data)

**Business Question:** "What operational incidents happened, where, and how severe were they?"

### Implementation

```sql
-- Aggregate to daily level by region and severity
SELECT 
  event_day,
  region_normalized as region,
  severity_band,
  
  -- Incident counts
  COUNT(*) as incident_count,
  SUM(CASE WHEN severity_band IN ('HIGH', 'CRITICAL') THEN 1 ELSE 0 END) as elevated_incident_count,
  SUM(CASE WHEN severity_band = 'CRITICAL' THEN 1 ELSE 0 END) as critical_incident_count,
  
  -- Duration metrics
  SUM(duration_minutes) as total_duration_minutes,
  AVG(duration_minutes) as avg_duration_minutes,
  MAX(duration_minutes) as max_duration_minutes,
  
  -- Customer impact
  SUM(affected_customers) as total_customers_affected,
  AVG(affected_customers) as avg_customers_per_incident,
  MAX(affected_customers) as max_customers_affected,
  
  COUNT(DISTINCT event_type) as unique_event_types
FROM silver_grid_events
GROUP BY event_day, region_normalized, severity_band
```

### Output: `gold_grid_incident_summary`

**Table:** `vattenfall_dev.gold.gold_grid_incident_summary`

**Grain:** `event_day` × `region` × `severity_band` (97 records from 165 events)

**Key Columns:**
* `event_day`, `region`, `severity_band`
* `incident_count`, `elevated_incident_count`, `critical_incident_count`
* `total_duration_minutes`, `avg_duration_minutes`, `max_duration_minutes`
* `total_customers_affected`, `avg_customers_per_incident`, `max_customers_affected`
* `unique_event_types`

### Key Findings

**Regional Distribution (165 total incidents):**
* Sweden: 60 incidents (36%) - 169K customers - 148 min avg duration
* Finland: 47 incidents (28%) - 115K customers - 116 min avg duration
* Norway: 30 incidents (18%) - 88K customers - 91 min avg duration
* Denmark: 28 incidents (17%) - 58K customers - 112 min avg duration

**Severity Distribution:**
* Critical Priority: 98 incidents (59%) - 360K customers (84% of impact)
* Medium Priority: 49 incidents (30%) - 53K customers - 73 min avg
* High Priority: 14 incidents (8%) - 17K customers - 228 min avg (longest restoration)
* Low Priority: 4 incidents (2%) - 243 customers

**Worst Day:** January 4, Norway - 22,067 customers affected

## Future Enhancement: Composite Condition Table

**Not Yet Implemented**

This would combine all three gold tables into a unified regional operational health score.

### Proposed Design: `gold_regional_condition_daily`

**Purpose:** Composite health score combining market, weather, and incident indicators

**Grain:** Date × Region

**Logic:**
```sql
-- 1. Normalize to 0-100
market_norm = (avg_price / max_regional_price) * 100
weather_norm = (weather_risk_score / 9.0) * 100  
incident_norm = (incident_count / max_daily_incidents) * 100

-- 2. Apply weights
risk_score = 
  (market_norm * 0.20) +    -- Market: 20%
  (weather_norm * 0.30) +   -- Weather: 30%
  (incident_norm * 0.50)    -- Incidents: 50%

-- 3. Invert (100 = healthy)
health_score = 100 - risk_score

-- 4. Categorize
operational_condition = 
  CASE
    WHEN health_score >= 85 THEN 'EXCELLENT'  -- 🟢
    WHEN health_score >= 70 THEN 'GOOD'       -- 🟢
    WHEN health_score >= 50 THEN 'FAIR'       -- 🟡
    WHEN health_score >= 30 THEN 'POOR'       -- 🟠
    ELSE 'CRITICAL'                           -- 🔴
  END
```

**Override Rule:** Force CRITICAL if any critical incidents occurred that day

### Alert Actions

| Condition | Action |
|-----------|--------|
| CRITICAL | Page COO/CEO, activate emergency ops |
| POOR | Notify VP Operations, mobilize crews |
| FAIR | Regional manager monitors |
| GOOD/EXCELLENT | Routine monitoring |

---

## Summary

**Current State:** Three foundational gold tables providing daily aggregations of market prices, weather conditions, and grid incidents.

**Next Step:** Build composite condition table to provide unified operational health scoring for executive dashboards.